In [86]:
import os
from dotenv import load_dotenv
from pathlib import Path
# from dataclasses import dataclass
from typing import List, Dict, Any

import xml.etree.ElementTree as ET

In [87]:

BASE_DIR = Path.cwd().parents[0]


load_dotenv(BASE_DIR / "creds" / ".env")




True

In [88]:
def xml_to_dict(element) -> Dict[str, Any]:
    """Convert XML element to clean dictionary."""
    result = {}
    
    # Add attributes if they exist
    if element.attrib:
        result['_attributes'] = element.attrib
    
    # Add text content
    if element.text and element.text.strip():
        result['_text'] = element.text.strip()
    
    # Process child elements
    for child in element:
        child_data = xml_to_dict(child)
        
        # Handle multiple elements with same tag
        if child.tag in result:
            if not isinstance(result[child.tag], list):
                result[child.tag] = [result[child.tag]]
            result[child.tag].append(child_data)
        else:
            # Single element - just store the data
            result[child.tag] = child_data if child_data else (child.text or None)
    
    return result if result else None


def parse_vehicle_search(xml_string: str) -> Dict:
    """Parse vehicle search response into clean data."""
    root = ET.fromstring(xml_string)
    
    # Extract header info
    header = root.find('Header')
    response_data = {
        'status': header.findtext('Status') if header is not None else None,
        'status_code': header.findtext('StatusCode') if header is not None else None,
        'vehicles': []
    }
    
    # Extract vehicle data
    for vehicle_elem in root.findall('.//VehicleSearchItem'):
        # Extract all links
        links = []
        links_container = vehicle_elem.find('Links')
        if links_container is not None:
            for link in links_container.findall('Link'):
                links.append({
                    'href': link.findtext('Href'),
                    'rel': link.findtext('Rel')
                })
        
        vehicle = {
            'base_vehicle_id': vehicle_elem.findtext('BaseVehicleID'),
            'make_name': vehicle_elem.findtext('MakeName'),
            'model_name': vehicle_elem.findtext('ModelName'),
            'sub_model_name': vehicle_elem.findtext('SubModelName'),
            'year': vehicle_elem.findtext('Year'),
            'engine_description': vehicle_elem.findtext('EngineDescription'),
            'vehicle_id': vehicle_elem.findtext('VehicleID'),
            'is_active': vehicle_elem.findtext('VehicleIsActive') == 'true',
            'links': links
        }
        response_data['vehicles'].append(vehicle)
    
    return response_data

<h2>Set Up</h2>
<p>
Run this first to set up the required functions:
</p>

In [89]:
from datetime import datetime, timezone
from urllib.parse import urlparse
import urllib
import hmac
import hashlib
import requests
import base64

C_PUBLIC_KEY = os.getenv("C_PUBLIC_KEY")
C_PRIVATE_KEY = os.getenv("C_PRIVATE_KEY")


if (not C_PUBLIC_KEY) | (not C_PRIVATE_KEY):
    raise EnvironmentError("API keys not set") 

def GenerateSharedAuth(d: datetime, public_key: str, private_key: str, uri: str, http_verb: str) -> bytes:
    utc_time = d.astimezone(timezone.utc)
    epoch_time = int(utc_time.timestamp())
    relative_url = urlparse(uri).path
    encoded_url = urllib.parse.quote(relative_url)

    plain_sig = public_key + chr(10) + http_verb + chr(10) + str(epoch_time) + chr(10) + encoded_url
    key = private_key.encode('ascii')
    byte_sig = plain_sig.encode('ascii')

    signature = hmac.new(key, byte_sig, hashlib.sha256).digest()
    return signature

def GetResponse(uri: str) -> str:
    today = datetime.now()
    auth = GenerateSharedAuth(today, C_PUBLIC_KEY, C_PRIVATE_KEY, uri, "GET")
    headers = {"Authorization": "Shared " + C_PUBLIC_KEY + ":" + base64.b64encode(auth).decode('ascii'), 
               "Date": today.astimezone(timezone.utc).strftime("%a, %d %b %Y %H:%M:%S GMT"),
               "Host": "api.motor.com"}

    response = requests.get("https://api.motor.com" + uri, headers=headers)
    return response.text

In [120]:


def extract_keywords_from_xml(xml_string: str, debug: bool = False) -> Dict:
    """
    Extract status, status_code, ApplicationID, and DisplayName from any XML response.
    Returns application_ids as a list of dicts with id and display_name.
    Handles nested ApplicationIDs and XML namespaces.
    Strips quotes from extracted values.
    """
    root = ET.fromstring(xml_string)

    # Strip namespace from tag names for easier searching
    def strip_ns(tag):
        return tag.split('}')[-1] if '}' in tag else tag

    def strip_quotes(value):
        """Remove surrounding quotes from string values."""
        if value and isinstance(value, str):
            return value.strip('\'"')
        return value

    # Extract status and status_code
    result = {
        'status': None,
        'status_code': None,
        'application_ids': []  # List of dicts with id and display_name
    }

    # Search for ApplicationID/DisplayName pairs within same parent
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag == 'Status' and elem.text:
            result['status'] = strip_quotes(elem.text)
        elif tag == 'StatusCode' and elem.text:
            result['status_code'] = strip_quotes(elem.text)
        elif tag == 'EstimatedWorkTimeApplicationSummary' or tag == 'ApplicationSummary':
            # Extract paired ApplicationID and DisplayName from same parent
            app_id_elem = next((e for e in elem if strip_ns(e.tag) == 'ApplicationID'), None)
            display_name_elem = next((e for e in elem if strip_ns(e.tag) == 'DisplayName'), None)

            if app_id_elem is not None and app_id_elem.text:
                app_id = strip_quotes(app_id_elem.text)
                display_name = strip_quotes(display_name_elem.text) if display_name_elem is not None and display_name_elem.text else None
                
                # Append as dict to list
                result['application_ids'].append({
                    'id': app_id,
                    'display_name': display_name if display_name else 'Unknown'
                })

    if debug:
        print(f"Found {len(result['application_ids'])} ApplicationID(s)")

    return result


def extract_application_id(xml_string: str) -> str:
    """
    Extract ApplicationID from any XML response.
    Returns the first ApplicationID found.
    """
    root = ET.fromstring(xml_string)
    
    def strip_ns(tag):
        return tag.split('}')[-1] if '}' in tag else tag
    
    # Find first ApplicationID element
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag == 'ApplicationID' and elem.text:
            return elem.text
    
    return None


def extract_all_part_application_ids(xml_string: str, debug: bool = False) -> List[str]:
    """
    Extract ALL part ApplicationIDs from a parts-summary response.
    Returns a list of all ApplicationID values found within PartApplicationSummary/PartApp containers.
    """
    root = ET.fromstring(xml_string)

    def strip_ns(tag):
        return tag.split('}')[-1] if '}' in tag else tag

    ids = []

    # Look for ApplicationID within PartApplicationSummary/PartApp containers
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag == 'PartApplicationSummary' or tag == 'PartApp':
            # Look for ApplicationID as direct child
            for child in elem:
                child_tag = strip_ns(child.tag)
                if child_tag == 'ApplicationID' and child.text:
                    ids.append(child.text)
                    if debug:
                        print(f"  Found ApplicationID: {child.text}")
                    break  # Only take first ApplicationID per container

    if debug:
        print(f"Total ApplicationIDs found: {len(ids)}")

    return ids


def extract_estimated_work_time(xml_string: str) -> Dict:
    """
    Extract EstimatedWorkTime details from XML response.
    Returns status, status_code, and a details dict with labor time and skill information.
    """
    root = ET.fromstring(xml_string)
    
    def strip_ns(tag):
        return tag.split('}')[-1] if '}' in tag else tag
    
    def get_text(elem, tag_name):
        """Helper to get text content from child element."""
        child = next((e for e in elem if strip_ns(e.tag) == tag_name), None)
        return child.text if child is not None and child.text else None
    
    result = {
        'status': None,
        'status_code': None,
        'details': {}
    }
    
    # Extract status and status_code
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag == 'Status' and elem.text:
            result['status'] = elem.text
        elif tag == 'StatusCode' and elem.text:
            result['status_code'] = elem.text
    
    # Find EstimatedWorkTime element
    work_time_elem = next((e for e in root.iter() if strip_ns(e.tag) == 'EstimatedWorkTime'), None)
    
    if work_time_elem is not None:
        # Extract Job Description from Notes > Note > Text
        job_description = None
        notes_elem = next((e for e in work_time_elem if strip_ns(e.tag) == 'Notes'), None)
        if notes_elem is not None:
            note_elem = next((e for e in notes_elem if strip_ns(e.tag) == 'Note'), None)
            if note_elem is not None:
                job_description = get_text(note_elem, 'Text')
        
        # Extract RequiredSkill > Description
        required_skill = None
        skill_elem = next((e for e in work_time_elem if strip_ns(e.tag) == 'RequiredSkill'), None)
        if skill_elem is not None:
            required_skill = get_text(skill_elem, 'Description')
        
        # Build details dict
        result['details'] = {
            'job_description': job_description,
            'additional_labor_time': get_text(work_time_elem, 'AdditionalLaborTime'),
            'additional_warranty_labor_time': get_text(work_time_elem, 'AdditionalWarrantyLaborTime'),
            'all_labor_time': get_text(work_time_elem, 'AllLaborTime'),
            'all_warranty_labor_time': get_text(work_time_elem, 'AllWarrantyLaborTime'),
            'base_labor_time': get_text(work_time_elem, 'BaseLaborTime'),
            'base_warranty_labor_time': get_text(work_time_elem, 'BaseWarrantyLaborTime'),
            'labor_time_interval': get_text(work_time_elem, 'LaborTimeInterval'),
            'required_skill': required_skill,
            'service_type': get_text(work_time_elem, 'ServiceType'),
            'base_labor_time_average': get_text(work_time_elem, 'BaseLaborTimeAverage'),
            'is_active': get_text(work_time_elem, 'IsActive'),
            'type': get_text(work_time_elem, 'Type')
        }
    
    return result


def extract_part_details(xml_string: str) -> Dict:
    """
    Extract part details from XML response.
    Returns status, status_code, and a details dict with part information.
    Handles nested structure: Part > PricingFamilies > PartPricingFamily > Pricing > PartPricing
    """
    root = ET.fromstring(xml_string)
    
    def strip_ns(tag):
        return tag.split('}')[-1] if '}' in tag else tag
    
    def get_text(elem, tag_name):
        """Helper to get text content from child element."""
        child = next((e for e in elem if strip_ns(e.tag) == tag_name), None)
        return child.text if child is not None and child.text else None
    
    result = {
        'status': None,
        'status_code': None,
        'details': {}
    }
    
    # Extract status and status_code
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag == 'Status' and elem.text:
            result['status'] = elem.text
        elif tag == 'StatusCode' and elem.text:
            result['status_code'] = elem.text
    
    # Find Part element
    part_elem = next((e for e in root.iter() if strip_ns(e.tag) == 'Part'), None)
    
    if part_elem is not None:
        part_number = get_text(part_elem, 'PartNumber')
        
        # Navigate to PartPricingFamily
        country_name = None
        manufacturer_name = None
        effective_date = None
        is_current = None
        oepr_part_number = None
        motor_part_number = None
        net_core_price = None
        category_name = None
        part_terminology_name = None
        price = None
        return_old_part = None
        
        pricing_families = next((e for e in part_elem if strip_ns(e.tag) == 'PricingFamilies'), None)
        if pricing_families is not None:
            part_pricing_family = next((e for e in pricing_families if strip_ns(e.tag) == 'PartPricingFamily'), None)
            if part_pricing_family is not None:
                # Extract CountryInfo > Name
                country_elem = next((e for e in part_pricing_family if strip_ns(e.tag) == 'CountryInfo'), None)
                if country_elem is not None:
                    country_name = get_text(country_elem, 'Name')
                
                # Extract ManufacturerInfo > Name
                manufacturer_elem = next((e for e in part_pricing_family if strip_ns(e.tag) == 'ManufacturerInfo'), None)
                if manufacturer_elem is not None:
                    manufacturer_name = get_text(manufacturer_elem, 'Name')
                
                # Navigate to Pricing > PartPricing
                pricing_elem = next((e for e in part_pricing_family if strip_ns(e.tag) == 'Pricing'), None)
                if pricing_elem is not None:
                    part_pricing = next((e for e in pricing_elem if strip_ns(e.tag) == 'PartPricing'), None)
                    if part_pricing is not None:
                        effective_date = get_text(part_pricing, 'EffectiveDate')
                        is_current = get_text(part_pricing, 'IsCurrent')
                        oepr_part_number = get_text(part_pricing, 'OEPRPartNumber')
                        motor_part_number = get_text(part_pricing, 'MOTORPartNumber')
                        net_core_price = get_text(part_pricing, 'NetCorePrice')
                        price = get_text(part_pricing, 'Price')
                        return_old_part = get_text(part_pricing, 'ReturnOldPart')
                        
                        # Extract Category > Name from PCDBPart
                        pcdb_part = next((e for e in part_pricing if strip_ns(e.tag) == 'PCDBPart'), None)
                        if pcdb_part is not None:
                            category_elem = next((e for e in pcdb_part if strip_ns(e.tag) == 'Category'), None)
                            if category_elem is not None:
                                category_name = get_text(category_elem, 'Name')
                            part_terminology_name = get_text(pcdb_part, 'PartTerminologyName')
        
        # Build details dict
        result['details'] = {
            'part_number': part_number,
            'country_name': country_name,
            'manufacturer_name': manufacturer_name,
            'effective_date': effective_date,
            'is_current': is_current,
            'oepr_part_number': oepr_part_number,
            'motor_part_number': motor_part_number,
            'net_core_price': net_core_price,
            'category_name': category_name,
            'part_terminology_name': part_terminology_name,
            'price': price,
            'return_old_part': return_old_part
        }
    
    return result

In [91]:
# # Test the function
# result = extract_keywords_from_xml(resp)

# print("Extracted Keywords:")
# print(f"  Status: {result['status']}")
# print(f"  Status Code: {result['status_code']}")
# print(f"  Application ID: {result['application_id']}")
# print(f"  All Application IDs: {result['application_ids']}")

# # Convert to JSON
# print("\nAs JSON:")
# print(json.dumps({
#     'status': result['status'],
#     'status_code': result['status_code'],
#     'application_id': result['application_id']
# }, indent=2))

<h2>Get Vehicle Info by VIN</h2>

<p>/v1/Information/Vehicles/Search/ByVIN?vin={VIN}</p>

In [92]:
vin = "1FTEW1E45KFB21693"

veh_vin = {
            "US":{""
                    "escape_2014": "3FA6P0HD1ER388009",
                    "escape_2020": "3FA6P0D9XLR115438"},
            "CA":{
                "escape_2014": "1FMCU9G97EUB92197",
                "escape_2025": "1FMCU9NZXSUA08739"}
            }

In [93]:

def step_1(Vin):
    resp = GetResponse(f"/v1/Information/Vehicles/Search/ByVIN?vin={Vin}")
    # resp = GetResponse(f"/v1/Information/Vehicles/Search/ByVIN?vin={vin}")

    data = parse_vehicle_search(resp)

    print(f"Response Status: {data['status']} ({data['status_code']})\n")
    return data

<h2>Get Summary</h2>
<p>/v1/Information/Vehicles/Attributes/BaseVehicleId/{VehId}/Content/Summaries/Of/EstimatedWorkTimes</p>

In [ ]:
def step2(data):

   vehicle_id = data["vehicles"][0]["base_vehicle_id"]
   system_id = "5"
   group_id = ""
   sub_group_id = ""
   # print(vehicle_id)
   
   resp = GetResponse(f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}/Content/Summaries/Of/EstimatedWorkTimes/?systemID={system_id}&groupID={group_id}&subGroupID={sub_group_id}")


   # print(resp)
   if not resp:
      raise ValueError("API response fails - step 2")
   
   data = extract_keywords_from_xml(resp)
   data["vehicle_id"] = vehicle_id
   return data




In [190]:
def step3(data):
    """
    REFACTORED: Extract ALL work items organized as list of dicts.
    Takes output from step2 (contains vehicle_id and application_ids list).
    
    For each work-time application, loops through ALL EstimatedWorkTime items
    and creates separate dict objects for each. Returns a list where each element
    is a dict mapping service name to list of work items.
    
    Output structure:
    [
        {
            'Rack & Pinion Assembly R&R': [
                {
                    'application_id': '244560030',
                    'vehicle_id': '85215',
                    'job_description': 'Includes: The removal...',
                    'base_labor_time': '0.4',
                    'base_labor_time_description': 'One Side',
                    'all_labor_time': '0.4',
                    'all_labor_time_description': 'One Side',
                    'all_warranty_labor_time': '0',
                    'base_warranty_labor_time': '0',
                    'additional_labor_time': '0',
                    'additional_labor_time_description': '',
                    'additional_warranty_labor_time': '0',
                    'estimated_work_time_id': '10613',
                    'labor_time_interval': 'Hours',
                    'required_skill': 'Requires a person...',
                    'service_type': 'Service',
                    'base_labor_time_average': '0',
                    'is_active': 'false',
                    'type': 'Main Operation'
                },
                {
                    'application_id': '244560030',
                    'vehicle_id': '85215',
                    'job_description': 'Includes: The removal...',
                    'base_labor_time': '0.7',
                    'base_labor_time_description': 'Both Sides',
                    ... (second work item)
                }
            ]
        },
        {
            'Steering Knuckle R&R': [
                { work_item_1 },
                { work_item_2 },
                ...
            ]
        }
    ]
    """
    def strip_ns(tag):
        """Remove XML namespace from tag."""
        return tag.split('}')[-1] if '}' in tag else tag
    
    def get_text(elem, tag_name):
        """Get text content from child element."""
        child = next((e for e in elem if strip_ns(e.tag) == tag_name), None)
        return child.text if child is not None and child.text else None
    
    vehicle_id = data["vehicle_id"]
    application_ids = data["application_ids"]  # List of {id, display_name}
    
    result_list = []  # Master list to hold all service dicts
    
    # Loop through each work-time application
    for app in application_ids:
        app_id = app["id"]
        display_name = app["display_name"]
        
        # Get the response which may contain multiple EstimatedWorkTime items
        resp = GetResponse(
            f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
            f"/Content/Details/Of/EstimatedWorkTimes/{app_id}"
        )
        
        root = ET.fromstring(resp)
        
        # Find the Items container with all EstimatedWorkTime elements
        items_container = None
        for elem in root.iter():
            if strip_ns(elem.tag) == 'Items':
                items_container = elem
                break
        
        # Collect all work items for this service
        work_items_list = []
        
        if items_container is not None:
            for work_time_elem in items_container:
                if strip_ns(work_time_elem.tag) == 'EstimatedWorkTime':
                    # Extract required skill description
                    required_skill = None
                    skill_elem = next((e for e in work_time_elem if strip_ns(e.tag) == 'RequiredSkill'), None)
                    if skill_elem is not None:
                        required_skill = get_text(skill_elem, 'Description')
                    
                    # Extract job description from Notes > Note > Text
                    job_description = None
                    notes_elem = next((e for e in work_time_elem if strip_ns(e.tag) == 'Notes'), None)
                    if notes_elem is not None:
                        note_elem = next((e for e in notes_elem if strip_ns(e.tag) == 'Note'), None)
                        if note_elem is not None:
                            job_description = get_text(note_elem, 'Text')
                    
                    # Create a dict for this work item with all fields
                    work_item = {
                        'application_id': app_id,
                        'vehicle_id': vehicle_id,
                        'job_description': job_description,
                        'additional_labor_time': get_text(work_time_elem, 'AdditionalLaborTime'),
                        'additional_labor_time_description': get_text(work_time_elem, 'AdditionalLaborTimeDescription'),
                        'additional_warranty_labor_time': get_text(work_time_elem, 'AdditionalWarrantyLaborTime'),
                        'all_labor_time': get_text(work_time_elem, 'AllLaborTime'),
                        'all_labor_time_description': get_text(work_time_elem, 'AllLaborTimeDescription'),
                        'all_warranty_labor_time': get_text(work_time_elem, 'AllWarrantyLaborTime'),
                        'base_labor_time': get_text(work_time_elem, 'BaseLaborTime'),
                        'base_labor_time_description': get_text(work_time_elem, 'BaseLaborTimeDescription'),
                        'base_warranty_labor_time': get_text(work_time_elem, 'BaseWarrantyLaborTime'),
                        'estimated_work_time_id': get_text(work_time_elem, 'EstimatedWorkTimeID'),
                        'labor_time_interval': get_text(work_time_elem, 'LaborTimeInterval'),
                        'required_skill': required_skill,
                        'service_type': get_text(work_time_elem, 'ServiceType'),
                        'base_labor_time_average': get_text(work_time_elem, 'BaseLaborTimeAverage'),
                        'is_active': get_text(work_time_elem, 'IsActive'),
                        'type': get_text(work_time_elem, 'Type')
                    }
                    
                    # Add this work item to the list
                    work_items_list.append(work_item)
        
        # Create dict: {service_name: [list of work items]}
        service_dict = {display_name: work_items_list}
        
        # Add to master list
        result_list.append(service_dict)
    
    return result_list

In [ ]:
def step_4(labor_items):
    """
    Enrich each work item with its parts data.
    Input: list of dicts where each dict is {service_name: [work_items]}
    Output: same structure but with 'parts' list added to each work item
    """
    enriched_list = []
    
    for service_dict in labor_items:
        enriched_service_dict = {}
        
        for display_name, work_items in service_dict.items():
            enriched_work_items = []
            
            for work_item in work_items:
                application_id = work_item['application_id']
                vehicle_id = work_item['vehicle_id']
                
                # Get parts summary for this work-time
                resp = GetResponse(
                    f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
                    f"/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/{application_id}"
                )
                
                # Extract all part application IDs
                part_app_ids = extract_all_part_application_ids(resp)
                
                # Get details for each part
                parts_list = []
                for part_app_id in part_app_ids:
                    resp = GetResponse(
                        f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
                        f"/Content/Details/Of/Parts/{part_app_id}"
                    )
                    part_details = extract_part_details(resp)
                    
                    part_info = part_details.get('details', {})
                    parts_list.append({
                        'part_app_id': part_app_id,
                        **part_info
                    })
                
                enriched_work_item = {
                    **work_item,
                    'parts': parts_list
                }
                
                enriched_work_items.append(enriched_work_item)
            
            enriched_service_dict[display_name] = enriched_work_items
        
        enriched_list.append(enriched_service_dict)
    
    return enriched_list

In [ ]:
"""
UPDATED step_5: Returns a LIST of part details (not a single dict)
This is critical - step_5 MUST return a list for the pipeline to work.
"""
def step_5(data):
    """
    Get details for ALL parts (loop through all part_app_ids).
    IMPORTANT: Returns a LIST of dicts, NOT a single dict.
    """
    vehicle_id = data.get("vehicle_id")
    part_app_ids = data.get("part_app_ids", [])
    
    # Initialize result as a LIST
    all_parts_list = []
    
    # Loop through each part ID and get its details
    for part_app_id in part_app_ids:
        try:
            resp = GetResponse(
                f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
                f"/Content/Details/Of/Parts/{part_app_id}"
            )

            part_details = extract_part_details(resp)
            
            # Append to list
            all_parts_list.append({
                'part_app_id': part_app_id,
                'part_data': part_details
            })
        except Exception as e:
            print(f"    Error getting part {part_app_id}: {e}")
    
    # CRITICAL: Return the list, not a single dict
    return all_parts_list



In [158]:
veh_data = step_1(veh_vin["US"]["escape_2020"])



Response Status: OK (200)



In [ ]:

veh_data
 

{'status': 'OK',
 'status_code': '200',
 'vehicles': [{'base_vehicle_id': '85215',
   'make_name': 'Ford',
   'model_name': 'Fusion',
   'sub_model_name': 'Titanium',
   'year': '2020',
   'engine_description': '2.0L L4 (9) Turbocharged GAS FI',
   'vehicle_id': '179683',
   'is_active': True,
   'links': [{'href': '/v1/Information/Vehicles/Attributes/BaseVehicleID/85215/BaseVehicle',
     'rel': 'BaseVehicleDetails'},
    {'href': '/v1/Information/Vehicles/Attributes/VehicleID/179683/Vehicle',
     'rel': 'VehicleDetails'}]}]}

In [ ]:

labour_summary_items = step2(veh_data)
 

In [161]:
labour_summary_items

{'status': 'OK',
 'status_code': '200',
 'application_ids': [{'id': '244560030',
   'display_name': 'Rack & Pinion Assembly R&R'},
  {'id': '244549997', 'display_name': 'Steering Knuckle R&R'},
  {'id': '244550120', 'display_name': 'Steering Knuckle R&R'},
  {'id': '244579908', 'display_name': 'Tie Rod R&R'},
  {'id': '244580012', 'display_name': 'Tie Rod R&R'}],
 'vehicle_id': '85215'}

In [191]:
labour_items = step3(labour_summary_items)

In [203]:
len(labour_items[4])

1

In [ ]:
resp = GetResponse(f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{85215}/Content/Details/Of/EstimatedWorkTimes/{244560030}")


In [ ]:

print(resp)
 

<?xml version="1.0"?>
<MWSInformationContentRs xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <Body>
    <Attributes />
    <EstimatedWorkTimes>
      <EstimatedWorkTimeApp>
        <ApplicationID>244580012</ApplicationID>
        <AttributeMappings />
        <Category>
          <Article>Unscheduled Maintenance Times</Article>
          <ID>24</ID>
          <Product>Estimating</Product>
          <ProductType>Estimating</ProductType>
        </Category>
        <ContentSilos>
          <ContentSilo>
            <ID>28</ID>
            <Name>Mechanical Repair Labor (GEN5)</Name>
            <SourceSilos>
              <SourceSilo>
                <ID>1</ID>
                <Name>GEN5 Labor</Name>
              </SourceSilo>
            </SourceSilos>
          </ContentSilo>
        </ContentSilos>
        <IsActive>true</IsActive>
        <Links>
          <Link>
            <Href>/v1/Information/Vehicles/Attributes/BaseVehicleI

In [168]:
resp_4 = step_4(labour_details)

KeyError: 'vehicle_id'

In [167]:
resp = GetResponse(f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{85215}/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/{244579908}")
 

In [165]:
print(resp)

<?xml version="1.0"?>
<MWSPartSummaryRs xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <Body>
    <Attributes>
      <Countries>
        <VehicleCountry>
          <Code>USA</Code>
          <CountryID>1</CountryID>
          <Name>United States</Name>
        </VehicleCountry>
      </Countries>
      <SubModels>
        <VehicleSubModel>
          <SubModelID>1</SubModelID>
          <SubModelName>S</SubModelName>
        </VehicleSubModel>
        <VehicleSubModel>
          <SubModelID>31</SubModelID>
          <SubModelName>SE</SubModelName>
        </VehicleSubModel>
        <VehicleSubModel>
          <SubModelID>643</SubModelID>
          <SubModelName>SEL</SubModelName>
        </VehicleSubModel>
        <VehicleSubModel>
          <SubModelID>1643</SubModelID>
          <SubModelName>Titanium</SubModelName>
        </VehicleSubModel>
        <VehicleSubModel>
          <SubModelID>2326</SubModelID>
          <SubModelName

In [ ]:


step_5(labour_more_details)

In [100]:
labour_details["application_id"]

'244560030'

In [101]:
labour_more_details = step_4(labour_details)

In [102]:
step_5(labour_more_details)

[{'part_app_id': '338614419',
  'part_data': {'status': 'OK',
   'status_code': '200',
   'details': {'part_number': 'KG9Z 3504-H',
    'country_name': 'United States',
    'manufacturer_name': 'Ford',
    'effective_date': '2026-04-01T00:00:00',
    'is_current': 'true',
    'oepr_part_number': 'KG9Z 3504-H',
    'motor_part_number': 'KG9Z3504H',
    'net_core_price': '0.00',
    'category_name': 'Steering',
    'part_terminology_name': 'Rack and Pinion Assembly',
    'price': '2718.18',
    'return_old_part': 'T'}}},
 {'part_app_id': '338614420',
  'part_data': {'status': 'OK',
   'status_code': '200',
   'details': {'part_number': 'KG9Z 3504-H',
    'country_name': 'United States',
    'manufacturer_name': 'Ford',
    'effective_date': '2026-04-01T00:00:00',
    'is_current': 'true',
    'oepr_part_number': 'KG9Z 3504-H',
    'motor_part_number': 'KG9Z3504H',
    'net_core_price': '0.00',
    'category_name': 'Steering',
    'part_terminology_name': 'Rack and Pinion Assembly',
    '

In [103]:
# def run_pipeline(vin: str) -> List[Dict]:
#     """
#     Full pipeline function: takes a VIN and returns all parts information 
#     for every work-time application and related part.
    
#     Returns a list of dicts, one entry per (work-time × part) combination.
#     Each entry contains work-time metadata, labor details, and part details.
#     """
#     # Step 1: VIN lookup
#     resp = GetResponse(f"/v1/Information/Vehicles/Search/ByVIN?vin={vin}")
#     vehicle_data = parse_vehicle_search(resp)
#     base_vehicle_id = vehicle_data["vehicles"][0]["base_vehicle_id"]
    
#     # Step 2: Get all work-time application IDs
#     resp = GetResponse(
#         f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{base_vehicle_id}"
#         f"/Content/Summaries/Of/EstimatedWorkTimes/?systemID=5&groupID=&subGroupID="
#     )
#     summaries = extract_keywords_from_xml(resp)
    
#     results = []
    
#     # Step 3: Loop over each work-time application
#     for app in summaries["application_ids"]:
#         app_id = app["id"]
#         display_name = app["display_name"]
        
#         # Get labor details
#         resp = GetResponse(
#             f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{base_vehicle_id}/Content/Details/Of/EstimatedWorkTimes/{app_id}"
#         )
#         labor = extract_estimated_work_time(resp)
        
#         # Get related part application IDs
#         resp = GetResponse(
#             f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{base_vehicle_id}/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/{app_id}"
#         )
#         part_app_ids = extract_all_part_application_ids(resp)
        
#         # Get details for each part
#         for part_app_id in part_app_ids:
#             resp = GetResponse(
#                 f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{base_vehicle_id}/Content/Details/Of/Parts/{part_app_id}"
#             )
#             part = extract_part_details(resp)
            
#             results.append({
#                 'work_time_application_id': app_id,
#                 'work_time_display_name': display_name,
#                 'labor_details': labor.get('details', {}),
#                 'part_details': part.get('details', {})
#             })
    
#     return results
# results = run_pipeline(veh_vin["US"]["escape_2020"])
# results

In [104]:
def run_pipeline_v2(vin: str) -> Dict:
    """
    Pipeline that follows the exact same strategy as step1→step2→step3→step4→step5.
    Loops through all application IDs and all parts, stores results by vehicle_id.
    Uses modified step_4 and step_5 that handle lists of part IDs.
    """
    print(f"Starting pipeline for VIN: {vin}\n")
    
    # STEP 1: Vehicle VIN lookup
    print("STEP 1: Vehicle lookup...")
    resp = GetResponse(f"/v1/Information/Vehicles/Search/ByVIN?vin={vin}")
    data = parse_vehicle_search(resp)
    print(f"✓ Vehicle found\n")
    
    # STEP 2: Get all work-time application IDs
    print("STEP 2: Get work-time summaries...")
    vehicle_id = data["vehicles"][0]["base_vehicle_id"]
    system_id = "5"
    resp = GetResponse(
        f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
        f"/Content/Summaries/Of/EstimatedWorkTimes/?systemID={system_id}&groupID=&subGroupID="
    )
    app_ID = extract_keywords_from_xml(resp)
    app_ID["vehicle_id"] = vehicle_id
    print(f"✓ Found {len(app_ID['application_ids'])} work-time applications\n")
    
    # Storage for all results organized by vehicle_id
    all_results = {
        vehicle_id: {
            'vehicle_info': data["vehicles"][0],
            'parts_data': []
        }
    }
    
    # STEP 3-5: Loop through EACH application_id and ALL its parts
    print("STEPS 3-5: Processing each application and its parts...\n")
    for app_idx, app in enumerate(app_ID['application_ids'], 1):
        application_id = app['id']
        display_name = app['display_name']
        
        print(f"  {app_idx}. Processing: {display_name} (ID: {application_id})")
        
        # STEP 3: Get labor details
        labour_details_data = {
            'vehicle_id': vehicle_id,
            'application_id': application_id,
            'display_name': display_name
        }
        resp = GetResponse(
            f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
            f"/Content/Details/Of/EstimatedWorkTimes/{application_id}"
        )
        labour_data = extract_estimated_work_time(resp)
        labour_details_data.update(labour_data)
        print(f"     ✓ Got labor details")
        
        # STEP 4: Get parts summary (get ALL part application IDs)
        step4_result = step_4(labour_details_data)
        part_app_ids = step4_result.get("part_app_ids", [])
        print(f"     ✓ Found {len(part_app_ids)} parts")
        
        # STEP 5: Loop through EACH part and get details
        parts_list = step_5(step4_result)
        
        # Ensure parts_list is actually a list
        if not isinstance(parts_list, list):
            print(f"     ⚠️  Step 5 returned {type(parts_list)}, expected list")
            parts_list = []
        
        for part_idx, part_result in enumerate(parts_list, 1):
            # Handle both old and new data structures
            if isinstance(part_result, dict):
                part_number = part_result.get('part_data', {}).get('details', {}).get('part_number', 'N/A')
                part_app_id = part_result.get('part_app_id')
                part_details = part_result.get('part_data', {}).get('details', {})
            else:
                part_number = 'N/A'
                part_app_id = 'N/A'
                part_details = {}
            
            # Store combined result
            result_entry = {
                'work_time': {
                    'application_id': application_id,
                    'display_name': display_name,
                    'labor_details': labour_data.get('details', {})
                },
                'part': {
                    'application_id': part_app_id,
                    'details': part_details
                }
            }
            all_results[vehicle_id]['parts_data'].append(result_entry)
            print(f"       {part_idx}. ✓ {part_number}")
        
        print()
    
    total_parts = len(all_results[vehicle_id]['parts_data'])
    print(f"\n✓ Pipeline complete!")
    print(f"  Vehicle ID: {vehicle_id}")
    print(f"  Total work-times: {len(app_ID['application_ids'])}")
    print(f"  Total parts: {total_parts}\n")
    
    return all_results

# Test with the working VIN
results_v2 = run_pipeline_v2(veh_vin["US"]["escape_2020"])
print(f"\nFinal result structure: {list(results_v2.keys())}")

Starting pipeline for VIN: 3FA6P0D9XLR115438

STEP 1: Vehicle lookup...
✓ Vehicle found

STEP 2: Get work-time summaries...
✓ Found 5 work-time applications

STEPS 3-5: Processing each application and its parts...

  1. Processing: Rack & Pinion Assembly R&R (ID: 244560030)
     ✓ Got labor details
     ✓ Found 8 parts
       1. ✓ KG9Z 3504-H
       2. ✓ KG9Z 3504-H
       3. ✓ KG9Z 3504-H
       4. ✓ KG9Z 3504-H
       5. ✓ KG9Z 3504-H
       6. ✓ KG9Z 3504-H
       7. ✓ KG9Z 3504-H
       8. ✓ KG9Z 3504-H

  2. Processing: Steering Knuckle R&R (ID: 244549997)
     ✓ Got labor details
     ✓ Found 24 parts
       1. ✓ NR
       2. ✓ NR
       3. ✓ NR
       4. ✓ NR
       5. ✓ NR
       6. ✓ NR
       7. ✓ NR
       8. ✓ NR
       9. ✓ HP5Z 3K186-A
       10. ✓ HP5Z 3K186-A
       11. ✓ HP5Z 3K186-A
       12. ✓ HP5Z 3K186-A
       13. ✓ HP5Z 3K186-A
       14. ✓ HP5Z 3K186-A
       15. ✓ HP5Z 3K186-A
       16. ✓ HP5Z 3K186-A
       17. ✓ HP5Z 3K185-A
       18. ✓ HP5Z 3K185-A
      

In [119]:
results_v2["85215"]["parts_data"]

[{'work_time': {'application_id': '244560030',
   'display_name': 'Rack & Pinion Assembly R&R',
   'labor_details': {'job_description': 'Includes: The removal of component and all necessary components for access, Time to add fluid to the system. Does not include: System diagnosis and testing, wheel alignment, or a vehicle road test.',
    'additional_labor_time': '0',
    'additional_warranty_labor_time': '0',
    'all_labor_time': '0',
    'all_warranty_labor_time': '0',
    'base_labor_time': '4.2',
    'base_warranty_labor_time': '0',
    'labor_time_interval': 'Hours',
    'required_skill': 'Requires a person with limited skills and wherever simpler measuring devices are used for proper repair (Feeler Gauge, Belt Tensioner Gauge,  etc). This person must have a thorough working knowledge of the component being serviced.',
    'service_type': 'Service',
    'base_labor_time_average': '4.27',
    'is_active': 'true',
    'type': 'Main Operation'}},
  'part': {'application_id': '338614

In [78]:
# CRITICAL DEBUG: Understand why extract_all_part_application_ids fails
print("="*80)
print("DEBUGGING: XML Element Structure Analysis")
print("="*80 + "\n")

vehicle_id = "85215"
application_id = "244560030"

resp = GetResponse(
    f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
    f"/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/{application_id}"
)

root = ET.fromstring(resp)

def strip_ns(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Step 1: Find all PartApplicationSummary elements
print("Step 1: Looking for PartApplicationSummary elements...\n")
part_app_summaries = []
for elem in root.iter():
    tag = strip_ns(elem.tag)
    if tag == 'PartApplicationSummary':
        part_app_summaries.append(elem)
        print(f"✓ Found PartApplicationSummary")
        print(f"  Full tag (with namespace): '{elem.tag}'")
        print(f"  Stripped tag: '{tag}'")
        
        # Check its children
        print(f"  Direct children:")
        child_count = 0
        for child in elem:
            child_count += 1
            child_tag = strip_ns(child.tag)
            print(f"    [{child_count}] {child_tag:30s} = {child.text[:50] if child.text else 'None'}")
            
            # If it's ApplicationID, show it
            if child_tag == 'ApplicationID':
                print(f"        → FOUND ApplicationID: {child.text}")
        
        if child_count == 0:
            print(f"    (no direct children)")
        print()

print(f"Total PartApplicationSummary elements found: {len(part_app_summaries)}\n")

# Step 2: Try the extraction with detailed steps
print("Step 2: Simulating extract_all_part_application_ids logic...\n")
ids = []
containers_found = 0
for elem in root.iter():
    tag = strip_ns(elem.tag)
    if tag == 'PartApplicationSummary' or tag == 'PartApp':
        containers_found += 1
        print(f"Found container: {tag}")
        
        # Look for ApplicationID child
        for e in elem:
            e_tag = strip_ns(e.tag)
            if e_tag == 'ApplicationID':
                print(f"  ✓ Found child ApplicationID: {e.text}")
                if e.text:
                    ids.append(e.text)
        
        # Also try with next() like in the function
        app_id_elem = next((e for e in elem if strip_ns(e.tag) == 'ApplicationID'), None)
        if app_id_elem is not None:
            print(f"  next() also found: {app_id_elem.text}")
        else:
            print(f"  next() returned None")

print(f"\nTotal containers found: {containers_found}")
print(f"Total IDs extracted: {ids}\n")

# Step 3: Show what extract_application_id would find
print("Step 3: What extract_application_id finds...\n")
app_ids = []
for elem in root.iter():
    tag = strip_ns(elem.tag)
    if tag == 'ApplicationID' and elem.text:
        app_ids.append(elem.text)
        print(f"Found ApplicationID: {elem.text}")

print(f"Total ApplicationID elements: {len(app_ids)}\n")

DEBUGGING: XML Element Structure Analysis

Step 1: Looking for PartApplicationSummary elements...

✓ Found PartApplicationSummary
  Full tag (with namespace): 'PartApplicationSummary'
  Stripped tag: 'PartApplicationSummary'
  Direct children:
    [1] ApplicationID                  = 338614419
        → FOUND ApplicationID: 338614419
    [2] AppRelationType                = 
          
    [3] AttributeMappings              = 
          
    [4] ContentSilos                   = 
          
    [5] DisplayName                    = Rack and Pinion Assembly
    [6] IsActive                       = true
    [7] Links                          = 
          
    [8] Position                       = 
          
    [9] Qualifiers                     = 
          
    [10] Taxonomy                       = 
          
    [11] PCDBPart                       = 
          

✓ Found PartApplicationSummary
  Full tag (with namespace): 'PartApplicationSummary'
  Stripped tag: 'PartApplicationSummary'

<h2>Get Details</h2>
<p>/Vehicles/Attributes/BaseVehicleId/{VehicleId}/Content/Details/Of/EstimatedWorkTimes/{ApplicationId}</p>

<h2>Whatever link you want</h2>

In [218]:
resp = GetResponse("/v1/Information/Vehicles/Attributes/BaseVehicleId/81898/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/185575044")
print(resp)

<?xml version="1.0"?>
<MWSPartSummaryRs xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <Body>
    <Attributes>
      <Countries>
        <VehicleCountry>
          <Code>USA</Code>
          <CountryID>1</CountryID>
          <Name>United States</Name>
        </VehicleCountry>
      </Countries>
      <Engines>
        <EngineInfo>
          <Aspiration>Turbocharged</Aspiration>
          <BlockType>V</BlockType>
          <CID>183</CID>
          <CylinderCC>2993</CylinderCC>
          <CylinderHeadType>DOHC</CylinderHeadType>
          <CylinderLiter>3.0</CylinderLiter>
          <Cylinders>6</Cylinders>
          <Description>3.0L V6 (1) Turbocharged DIESEL FI</Description>
          <Designation>-</Designation>
          <EngineBoreInch>3.31</EngineBoreInch>
          <EngineBoreMetric>84.0</EngineBoreMetric>
          <EngineID>13215</EngineID>
          <EngineStrokeInch>3.54</EngineStrokeInch>
          <EngineStrokeMetric>

In [ ]:
# CRITICAL DEBUG: Understand why extract_all_part_application_ids fails
print("="*80)
print("DEBUGGING: XML Element Structure Analysis")
print("="*80 + "\n")

vehicle_id = "85215"
application_id = "244560030"

resp = GetResponse(
    f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
    f"/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/{application_id}"
)

root = ET.fromstring(resp)

def strip_ns(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Step 1: Find all PartApplicationSummary elements
print("Step 1: Looking for PartApplicationSummary elements...\n")
part_app_summaries = []
for elem in root.iter():
    tag = strip_ns(elem.tag)
    if tag == 'PartApplicationSummary':
        part_app_summaries.append(elem)
        print(f"✓ Found PartApplicationSummary")
        print(f"  Full tag (with namespace): '{elem.tag}'")
        print(f"  Stripped tag: '{tag}'")
        
        # Check its children
        print(f"  Direct children:")
        child_count = 0
        for child in elem:
            child_count += 1
            child_tag = strip_ns(child.tag)
            print(f"    [{child_count}] {child_tag:30s} = {child.text[:50] if child.text else 'None'}")
            
            # If it's ApplicationID, show it
            if child_tag == 'ApplicationID':
                print(f"        → FOUND ApplicationID: {child.text}")
        
        if child_count == 0:
            print(f"    (no direct children)")
        print()

print(f"Total PartApplicationSummary elements found: {len(part_app_summaries)}\n")

# Step 2: Try the extraction with detailed steps
print("Step 2: Simulating extract_all_part_application_ids logic...\n")
ids = []
containers_found = 0
for elem in root.iter():
    tag = strip_ns(elem.tag)
    if tag == 'PartApplicationSummary' or tag == 'PartApp':
        containers_found += 1
        print(f"Found container: {tag}")
        
        # Look for ApplicationID child
        for e in elem:
            e_tag = strip_ns(e.tag)
            if e_tag == 'ApplicationID':
                print(f"  ✓ Found child ApplicationID: {e.text}")
                if e.text:
                    ids.append(e.text)
        
        # Also try with next() like in the function
        app_id_elem = next((e for e in elem if strip_ns(e.tag) == 'ApplicationID'), None)
        if app_id_elem is not None:
            print(f"  next() also found: {app_id_elem.text}")
        else:
            print(f"  next() returned None")

print(f"\nTotal containers found: {containers_found}")
print(f"Total IDs extracted: {ids}\n")

# Step 3: Show what extract_application_id would find
print("Step 3: What extract_application_id finds...\n")
app_ids = []
for elem in root.iter():
    tag = strip_ns(elem.tag)
    if tag == 'ApplicationID' and elem.text:
        app_ids.append(elem.text)
        print(f"Found ApplicationID: {elem.text}")

print(f"Total ApplicationID elements: {len(app_ids)}\n")

In [206]:
def run_pipeline(vin: str):
    """
    Complete pipeline: VIN → All work-time applications and their parts.
    
    Returns a list where each element is a dict mapping service name to work items,
    and each work item contains all labor details plus a 'parts' list.
    
    Output structure:
    [
        {
            'Rack & Pinion Assembly R&R': [
                {
                    'application_id': '244560030',
                    'vehicle_id': '85215',
                    'job_description': '...',
                    'base_labor_time': '0.4',
                    ... (all 20 labor fields),
                    'parts': [
                        {
                            'part_app_id': '338614419',
                            'part_number': 'KG9Z 3504-H',
                            'price': '2718.18',
                            'manufacturer_name': 'Ford',
                            ... (all part detail fields)
                        },
                        ...
                    ]
                },
                { second_work_item_with_parts },
                ...
            ]
        },
        {
            'Steering Knuckle R&R': [ ... ],
            ...
        }
    ]
    """
    print(f"Starting pipeline for VIN: {vin}")
    
    # STEP 1: Vehicle lookup
    print("STEP 1: Vehicle lookup...")
    vehicle_data = step_1(vin)
    print(f"✓ Found: {vehicle_data['vehicles'][0]['make_name']} {vehicle_data['vehicles'][0]['model_name']} {vehicle_data['vehicles'][0]['year']}")
    
    # STEP 2: Get all work-time application IDs
    print("STEP 2: Get work-time summaries...")
    summaries = step2(vehicle_data)
    num_apps = len(summaries['application_ids'])
    print(f"✓ Found {num_apps} work-time applications")
    
    # STEP 3: Extract all work items from each application
    print("STEP 3: Extract work items...")
    labor_items = step3(summaries)
    print(f"✓ Processed {len(labor_items)} services")
    
    # STEP 4: Enrich each work item with its parts
    print("STEP 4: Enrich with parts data...")
    complete_data = step_4(labor_items)
    
    # Count total parts
    total_parts = 0
    for service_dict in complete_data:
        for service_name, work_items in service_dict.items():
            for work_item in work_items:
                total_parts += len(work_item.get('parts', []))
    
    print(f"✓ Found {total_parts} total parts")
    print(f"✓ Pipeline complete!")
    
    return complete_data


# Test with the known working VIN
print("" + "="*80)
print("RUNNING FULL PIPELINE")
print("="*80 + "")

results = run_pipeline(veh_vin["US"]["escape_2020"])

# Display summary
print("FINAL RESULT STRUCTURE:")
for service_dict in results:
    for service_name, work_items in service_dict.items():
        print(f"  {service_name}: {len(work_items)} work item(s)")
        for work_item in work_items:
            num_parts = len(work_item.get('parts', []))
            print(f"    - {work_item['job_description'][:50]}... ({num_parts} parts)")


RUNNING FULL PIPELINE
Starting pipeline for VIN: 3FA6P0D9XLR115438
STEP 1: Vehicle lookup...
Response Status: OK (200)

✓ Found: Ford Fusion 2020
STEP 2: Get work-time summaries...
✓ Found 5 work-time applications
STEP 3: Extract work items...
✓ Processed 5 services
STEP 4: Enrich with parts data...


AttributeError: 'list' object has no attribute 'items'

In [204]:

# # Test the complete pipeline
# if __name__ != "__main__":
#     print("
# " + "="*80)
#     print("TESTING COMPLETE PIPELINE")
#     print("="*80 + "
# ")
    
#     try:
#         # Run the pipeline with the test VIN
#         test_vin = veh_vin["US"]["escape_2020"]
#         print(f"Running pipeline for VIN: {test_vin}
# ")
        
#         results = run_pipeline(test_vin)
        
#         # Show summary
#         print("
# " + "="*80)
#         print("PIPELINE RESULTS SUMMARY")
#         print("="*80 + "
# ")
        
#         total_services = 0
#         total_work_items = 0
#         total_parts = 0
        
#         for service_dict in results:
#             for service_name, work_items in service_dict.items():
#                 total_services += 1
#                 total_work_items += len(work_items)
#                 for work_item in work_items:
#                     parts = work_item.get('parts', [])
#                     total_parts += len(parts)
#                     print(f"  Service: {service_name}")
#                     print(f"    Work Item ID: {work_item.get('application_id')}")
#                     print(f"    Description: {work_item.get('job_description', 'N/A')[:60]}...")
#                     print(f"    Parts: {len(parts)}")
#                     if parts:
#                         print(f"      First part: {parts[0].get('part_number')} ({parts[0].get('manufacturer_name')})")
#                     print()
        
#         print(f"
# FINAL SUMMARY:")
#         print(f"  Total services: {total_services}")
#         print(f"  Total work items: {total_work_items}")
#         print(f"  Total parts: {total_parts}")
        
#     except Exception as e:
#         import traceback
#         print(f"ERROR running pipeline: {e}")
#         traceback.print_exc()
